# RQ4: Compounding Risk Analysis

**Research Question**  
Do individuals with multiple risk factors exhibit diabetes rates statistically higher than those with only one risk factor?

- H4₀: No significant compounding effect (additive relationship)  
- H4₁: Multiple risks show synergistic effect (multiplicative relationship)

## 1. Data Loading

In [1]:
from pathlib import Path
from typing import Union, Optional
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from statsmodels.stats.proportion import proportions_ztest
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score
from sklearn.preprocessing import StandardScaler
from itertools import combinations
import json
import shap
import warnings
import logging


warnings.filterwarnings("ignore")

# Plotting style
plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")

# Configure Logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

In [2]:
BASE_DIR = Path.cwd().parent 
PROCESSED_DATA_DIR = BASE_DIR / "data_processed"
REPORTS_DIR = BASE_DIR / "reports"
CONFIG_DIR = BASE_DIR / "config"
BRFSS_CLEAN_ZIP_FILE = PROCESSED_DATA_DIR / "BRFSS_2015_2024_cleaned.zip"

# Ensure directories exist
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
def read_zipped_csv(
    zip_path: Union[str, Path],
    **read_csv_kwargs
) -> pd.DataFrame:
    """
    Read a single-CSV .zip file into a pandas DataFrame.

    Parameters
    ----------
    zip_path : str or Path
        Path to the .zip file (containing exactly one CSV, or a CSV
        with the same name as the zip).
    **read_csv_kwargs :
        Any extra keyword args passed through to pandas.read_csv
        (e.g. sep, dtype, parse_dates).

    Returns
    -------
    pd.DataFrame
    """
    zip_path = Path(zip_path)
    # For a single CSV in the zip, this is enough:
    return pd.read_csv(zip_path, compression="zip", **read_csv_kwargs)

In [4]:
df_raw = read_zipped_csv(BRFSS_CLEAN_ZIP_FILE)
logging.info(f"Initial data loaded with {df_raw.shape[0]} rows. and columns: {df_raw.shape[1]}")

2026-02-11 00:56:43 - INFO - Initial data loaded with 3101950 rows. and columns: 19


## 2. Risk Index Creation

Create risk index based on key modifiable risk factors:
1. Obesity (BMICAT = 3: Obese)
2. No Exercise (EXERCISE = 1: No)
3. Current Smoker (SMOKER = 1: Yes)
4. Heavy Alcohol (HEAVY_ALCOHOL_CONSUMPTION = 1: Yes)
5. Poor Health Status (HEALTH_STATUS = 0: Poor)

In [5]:
def create_risk_index(df):
    """
    Create risk index based on key modifiable risk factors
    """
    df = df.copy()
    
    # Define risk conditions
    df['RISK_OBESITY'] = (df['BMICAT'] == 3).astype(int)
    df['RISK_NO_EXERCISE'] = (df['EXERCISE'] == 1).astype(int)
    df['RISK_SMOKER'] = (df['SMOKER'] == 1).astype(int)
    df['RISK_HEAVY_ALCOHOL'] = (df['HEAVY_ALCOHOL_CONSUMPTION'] == 1).astype(int)
    df['RISK_POOR_HEALTH'] = (df['HEALTH_STATUS'] == 0).astype(int)
    
    # Calculate total risk score
    risk_cols = ['RISK_OBESITY', 'RISK_NO_EXERCISE', 'RISK_SMOKER',
                 'RISK_HEAVY_ALCOHOL', 'RISK_POOR_HEALTH']
    df['RISK_INDEX'] = df[risk_cols].sum(axis=1)
    
    # Create risk categories
    df['RISK_CATEGORY'] = pd.cut(df['RISK_INDEX'],
                                   bins=[-0.5, 0.5, 1.5, 2.5, 5.5],
                                   labels=['0 Risks', '1 Risk', '2 Risks', '3+ Risks'])
    
    # Filter valid diabetes data
    df = df[df['DIABETES'].isin([0, 1])]
    
    return df, risk_cols

## 3. Prevalence Calculation Functions

In [6]:
def calculate_prevalence_by_risk_level(df):
    """Calculate diabetes prevalence by risk index level"""
    results = []
    
    for risk_level in range(0, 6):  # 0 to 5 risks
        risk_data = df[df['RISK_INDEX'] == risk_level]
        
        if len(risk_data) > 0:
            total = len(risk_data)
            diabetic = (risk_data['DIABETES'] == 1).sum()
            prevalence = (diabetic / total * 100) if total > 0 else 0
            
            # 95% Confidence Interval (Wilson score interval)
            if total > 0:
                p = diabetic / total
                z = 1.96
                denominator = 1 + z**2 / total
                center = (p + z**2 / (2 * total)) / denominator
                margin = z * np.sqrt((p * (1 - p) / total + z**2 / (4 * total**2))) / denominator
                ci_lower = max(0, (center - margin) * 100)
                ci_upper = min(100, (center + margin) * 100)
            else:
                ci_lower = ci_upper = 0
            
            results.append({
                'Risk_Index': risk_level,
                'Total_N': total,
                'Diabetic_N': diabetic,
                'Prevalence_%': prevalence,
                'CI_Lower': ci_lower,
                'CI_Upper': ci_upper
            })
    
    return pd.DataFrame(results)

In [7]:
def calculate_prevalence_by_risk_category(df):
    """Calculate prevalence by risk category"""
    results = []
    
    for category in ['0 Risks', '1 Risk', '2 Risks', '3+ Risks']:
        cat_data = df[df['RISK_CATEGORY'] == category]
        
        if len(cat_data) > 0:
            total = len(cat_data)
            diabetic = (cat_data['DIABETES'] == 1).sum()
            prevalence = (diabetic / total * 100) if total > 0 else 0
            
            # 95% CI (Wilson score interval)
            if total > 0:
                p = diabetic / total
                z = 1.96
                denominator = 1 + z**2 / total
                center = (p + z**2 / (2 * total)) / denominator
                margin = z * np.sqrt((p * (1 - p) / total + z**2 / (4 * total**2))) / denominator
                ci_lower = max(0, (center - margin) * 100)
                ci_upper = min(100, (center + margin) * 100)
            else:
                ci_lower = ci_upper = 0
            
            results.append({
                'Risk_Category': category,
                'Total_N': total,
                'Diabetic_N': diabetic,
                'Prevalence_%': prevalence,
                'CI_Lower': ci_lower,
                'CI_Upper': ci_upper
            })
    
    return pd.DataFrame(results)

## 4. Compounding Effect Analysis

Test if compounding effect is additive or multiplicative by comparing observed vs expected prevalence.

In [8]:
def test_compounding_effect(df):
    """
    Test if compounding effect is additive or multiplicative
    Compare observed vs expected (additive) prevalence
    """
    # Get single risk prevalence (baseline)
    single_risk = df[df['RISK_INDEX'] == 1]
    single_prev = (single_risk['DIABETES'] == 1).sum() / len(single_risk) if len(single_risk) > 0 else 0
    
    # Get zero risk prevalence
    zero_risk = df[df['RISK_INDEX'] == 0]
    zero_prev = (zero_risk['DIABETES'] == 1).sum() / len(zero_risk) if len(zero_risk) > 0 else 0
    
    # Incremental risk per factor
    incremental_risk = single_prev - zero_prev
    
    results = []
    
    for risk_level in range(0, 6):
        risk_data = df[df['RISK_INDEX'] == risk_level]
        
        if len(risk_data) > 0:
            # Observed prevalence
            observed_prev = (risk_data['DIABETES'] == 1).sum() / len(risk_data)
            
            # Expected prevalence (additive model)
            expected_additive = zero_prev + (risk_level * incremental_risk)
            expected_additive = min(expected_additive, 1.0)  # Cap at 100%
            
            # Expected prevalence (multiplicative model)
            if risk_level > 0 and zero_prev > 0:
                odds_ratio = (single_prev / (1 - single_prev)) / (zero_prev / (1 - zero_prev))
                expected_odds = (zero_prev / (1 - zero_prev)) * (odds_ratio ** risk_level)
                expected_multiplicative = expected_odds / (1 + expected_odds)
            else:
                expected_multiplicative = zero_prev
            
            results.append({
                'Risk_Index': risk_level,
                'N': len(risk_data),
                'Observed_Prevalence': observed_prev * 100,
                'Expected_Additive_%': expected_additive * 100,
                'Expected_Multiplicative_%': expected_multiplicative * 100,
                'Additive_Diff': (observed_prev - expected_additive) * 100,
                'Multiplicative_Diff': (observed_prev - expected_multiplicative) * 100
            })
    
    return pd.DataFrame(results)

## 5. Statistical Testing

Perform pairwise comparisons between risk categories.

In [9]:
def perform_pairwise_comparisons(df):
    """Perform pairwise statistical tests between risk categories"""
    categories = ['0 Risks', '1 Risk', '2 Risks', '3+ Risks']
    results = []
    
    for i in range(len(categories)):
        for j in range(i + 1, len(categories)):
            cat1 = categories[i]
            cat2 = categories[j]
            
            data1 = df[df['RISK_CATEGORY'] == cat1]
            data2 = df[df['RISK_CATEGORY'] == cat2]
            
            count1 = (data1['DIABETES'] == 1).sum()
            count2 = (data2['DIABETES'] == 1).sum()
            nobs1 = len(data1)
            nobs2 = len(data2)
            
            if nobs1 > 0 and nobs2 > 0:
                # Two-proportion z-test
                counts = np.array([count1, count2])
                nobs = np.array([nobs1, nobs2])
                z_stat, p_value = proportions_ztest(counts, nobs, alternative='two-sided')
                
                prev1 = (count1 / nobs1 * 100) if nobs1 > 0 else 0
                prev2 = (count2 / nobs2 * 100) if nobs2 > 0 else 0
                
                # Odds ratio
                if count1 > 0 and count2 > 0:
                    odds1 = count1 / (nobs1 - count1)
                    odds2 = count2 / (nobs2 - count2)
                    odds_ratio = odds2 / odds1 if odds1 > 0 else np.nan
                else:
                    odds_ratio = np.nan
                
                results.append({
                    'Comparison': f'{cat1} vs {cat2}',
                    'Cat1': cat1,
                    'Cat1_Prevalence_%': prev1,
                    'Cat2': cat2,
                    'Cat2_Prevalence_%': prev2,
                    'Odds_Ratio': odds_ratio,
                    'Z_Statistic': z_stat,
                    'P_Value': p_value,
                    'Significant_at_0.05': p_value < 0.05
                })
    
    return pd.DataFrame(results)

## 6. Risk Factor Interactions

Analyze specific risk factor combinations to detect synergistic effects.

In [10]:
def analyze_risk_interactions(df, risk_cols):
    """Analyze specific risk factor combinations"""
    results = []
    
    for risk1, risk2 in combinations(risk_cols, 2):
        # Both risks present
        both = df[(df[risk1] == 1) & (df[risk2] == 1)]
        # Only risk1
        only1 = df[(df[risk1] == 1) & (df[risk2] == 0)]
        # Only risk2
        only2 = df[(df[risk1] == 0) & (df[risk2] == 1)]
        # Neither
        neither = df[(df[risk1] == 0) & (df[risk2] == 0)]
        
        if len(both) > 10 and len(neither) > 10:  # Minimum sample size
            prev_both = (both['DIABETES'] == 1).sum() / len(both) * 100
            prev_only1 = (only1['DIABETES'] == 1).sum() / len(only1) * 100 if len(only1) > 0 else 0
            prev_only2 = (only2['DIABETES'] == 1).sum() / len(only2) * 100 if len(only2) > 0 else 0
            prev_neither = (neither['DIABETES'] == 1).sum() / len(neither) * 100
            
            # Expected additive
            expected = prev_only1 + prev_only2 - prev_neither
            
            # Synergy index
            synergy = prev_both - expected
            
            results.append({
                'Risk1': risk1,
                'Risk2': risk2,
                'N_Both': len(both),
                'Prev_Both': prev_both,
                'Prev_Only1': prev_only1,
                'Prev_Only2': prev_only2,
                'Prev_Neither': prev_neither,
                'Expected_Additive': expected,
                'Synergy_Index': synergy,
                'Synergistic': synergy > 0
            })
    
    return pd.DataFrame(results).sort_values('Synergy_Index', ascending=False)

## 7. Visualization Functions

In [11]:
def plot_risk_index_analysis(prevalence_df, compound_df, output_path):
    """Visualize risk index analysis"""
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('RQ4: Compounding Risk Analysis',
                 fontsize=16, fontweight='bold')
    
    # Plot 1: Prevalence by risk index
    ax1 = axes[0, 0]
    ax1.plot(prevalence_df['Risk_Index'], prevalence_df['Prevalence_%'],
             marker='o', linewidth=3, markersize=10, color='darkred')
    ax1.fill_between(prevalence_df['Risk_Index'],
                      prevalence_df['CI_Lower'],
                      prevalence_df['CI_Upper'],
                      alpha=0.3, color='red')
    ax1.set_xlabel('Number of Risk Factors', fontsize=12)
    ax1.set_ylabel('Diabetes Prevalence (%)', fontsize=12)
    ax1.set_title('Prevalence by Risk Index', fontsize=14, fontweight='bold')
    ax1.grid(True, alpha=0.3)
    ax1.set_xticks(prevalence_df['Risk_Index'])
    
    # Plot 2: Observed vs Expected (Additive vs Multiplicative)
    ax2 = axes[0, 1]
    ax2.plot(compound_df['Risk_Index'], compound_df['Observed_Prevalence'],
             marker='o', label='Observed', linewidth=2, markersize=8)
    ax2.plot(compound_df['Risk_Index'], compound_df['Expected_Additive_%'],
             marker='s', linestyle='--', label='Expected (Additive)', linewidth=2)
    ax2.plot(compound_df['Risk_Index'], compound_df['Expected_Multiplicative_%'],
             marker='^', linestyle='--', label='Expected (Multiplicative)', linewidth=2)
    ax2.set_xlabel('Number of Risk Factors', fontsize=12)
    ax2.set_ylabel('Diabetes Prevalence (%)', fontsize=12)
    ax2.set_title('Observed vs Expected Models', fontsize=14, fontweight='bold')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # Plot 3: Sample size distribution
    ax3 = axes[1, 0]
    ax3.bar(prevalence_df['Risk_Index'], prevalence_df['Total_N'],
            color='steelblue', alpha=0.7)
    ax3.set_xlabel('Number of Risk Factors', fontsize=12)
    ax3.set_ylabel('Sample Size (N)', fontsize=12)
    ax3.set_title('Sample Distribution by Risk Index', fontsize=14, fontweight='bold')
    ax3.grid(True, alpha=0.3, axis='y')
    ax3.set_xticks(prevalence_df['Risk_Index'])
    
    # Add sample size labels
    for idx, row in prevalence_df.iterrows():
        ax3.text(row['Risk_Index'], row['Total_N'] + max(prevalence_df['Total_N'])*0.02,
                f"{row['Total_N']:,}", ha='center', va='bottom', fontsize=9)
    
    # Plot 4: Deviation from additive model
    ax4 = axes[1, 1]
    deviations = compound_df['Additive_Diff']
    colors = ['green' if x > 0 else 'red' for x in deviations]
    ax4.bar(compound_df['Risk_Index'], deviations, color=colors, alpha=0.7)
    ax4.axhline(y=0, color='black', linestyle='-', linewidth=1)
    ax4.set_xlabel('Number of Risk Factors', fontsize=12)
    ax4.set_ylabel('Deviation from Additive Model (%)', fontsize=12)
    ax4.set_title('Synergistic Effect (Observed - Expected)', fontsize=14, fontweight='bold')
    ax4.grid(True, alpha=0.3, axis='y')
    ax4.set_xticks(compound_df['Risk_Index'])
    
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f"✓ Plot saved: {output_path}")
    plt.close()

In [12]:
def plot_pairwise_comparisons(comparison_df, category_df, output_path):
    """Visualize pairwise comparisons"""
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('RQ4: Statistical Comparisons Between Risk Categories',
                 fontsize=16, fontweight='bold')
    
    # Plot 1: Prevalence by category with error bars
    ax1 = axes[0, 0]
    categories = category_df['Risk_Category']
    prevalence = category_df['Prevalence_%']
    ci_lower = category_df['CI_Lower']
    ci_upper = category_df['CI_Upper']
    
    x_pos = np.arange(len(categories))
    bars = ax1.bar(x_pos, prevalence, color=['green', 'yellow', 'orange', 'red'], alpha=0.7)
    errors = [prevalence - ci_lower, ci_upper - prevalence]
    ax1.errorbar(x_pos, prevalence, yerr=errors, fmt='none',
                 ecolor='black', capsize=5, capthick=2)
    ax1.set_xlabel('Risk Category', fontsize=12)
    ax1.set_ylabel('Diabetes Prevalence (%)', fontsize=12)
    ax1.set_title('Prevalence by Risk Category (with 95% CI)', fontsize=14, fontweight='bold')
    ax1.set_xticks(x_pos)
    ax1.set_xticklabels(categories, rotation=15)
    ax1.grid(True, alpha=0.3, axis='y')
    
    # Add values on bars
    for bar, val in zip(bars, prevalence):
        ax1.text(bar.get_x() + bar.get_width()/2, val + 1,
                f'{val:.1f}%', ha='center', va='bottom', fontweight='bold')
    
    # Plot 2: Odds ratios
    ax2 = axes[0, 1]
    comparisons = comparison_df['Comparison'].str.replace(' vs ', '\nvs\n')
    odds_ratios = comparison_df['Odds_Ratio']
    x_pos_comp = np.arange(len(comparisons))
    ax2.bar(x_pos_comp, odds_ratios, color='steelblue', alpha=0.7)
    ax2.axhline(y=1, color='red', linestyle='--', linewidth=2, label='No Effect (OR=1)')
    ax2.set_xlabel('Comparison', fontsize=12)
    ax2.set_ylabel('Odds Ratio', fontsize=12)
    ax2.set_title('Odds Ratios Between Risk Categories', fontsize=14, fontweight='bold')
    ax2.set_xticks(x_pos_comp)
    ax2.set_xticklabels(comparisons, fontsize=8)
    ax2.legend()
    ax2.grid(True, alpha=0.3, axis='y')
    
    # Plot 3: P-values
    ax3 = axes[1, 0]
    p_values = comparison_df['P_Value']
    colors_p = ['green' if p < 0.05 else 'red' for p in p_values]
    ax3.bar(x_pos_comp, p_values, color=colors_p, alpha=0.7)
    ax3.axhline(y=0.05, color='black', linestyle='--', linewidth=2, label='α = 0.05')
    ax3.set_xlabel('Comparison', fontsize=12)
    ax3.set_ylabel('P-Value', fontsize=12)
    ax3.set_title('Statistical Significance', fontsize=14, fontweight='bold')
    ax3.set_xticks(x_pos_comp)
    ax3.set_xticklabels(comparisons, fontsize=8)
    ax3.set_yscale('log')
    ax3.legend()
    ax3.grid(True, alpha=0.3, axis='y')
    
    # Plot 4: Significance summary
    ax4 = axes[1, 1]
    sig_count = comparison_df['Significant_at_0.05'].sum()
    not_sig = len(comparison_df) - sig_count
    labels = ['Significant\n(p < 0.05)', 'Not Significant\n(p ≥ 0.05)']
    sizes = [sig_count, not_sig]
    colors_pie = ['#2ecc71', '#e74c3c']
    explode = (0.1, 0)
    ax4.pie(sizes, explode=explode, labels=labels, autopct='%1.0f%%',
            startangle=90, colors=colors_pie, textprops={'fontsize': 12})
    ax4.set_title(f'Significance Summary\n({len(comparison_df)} comparisons)',
                  fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f"✓ Plot saved: {output_path}")
    plt.close()

## 8. Main Analysis Execution

### 8.1 Load and Prepare Data

In [13]:
print("="*70)
print("RQ4: COMPOUNDING RISK ANALYSIS")
print("="*70)

# Load data
print("\n[1] Loading data...")
df = df_raw.copy()
print(f"✓ Loaded {len(df):,} records")

RQ4: COMPOUNDING RISK ANALYSIS

[1] Loading data...
✓ Loaded 3,101,950 records


### 8.2 Create Risk Index

In [14]:
print("\n[2] Creating risk index...")
df, risk_cols = create_risk_index(df)
print(f"✓ Risk factors considered:")
for col in risk_cols:
    print(f"   - {col}")

print(f"\nRisk index distribution:")
print(df['RISK_INDEX'].value_counts().sort_index())


[2] Creating risk index...
✓ Risk factors considered:
   - RISK_OBESITY
   - RISK_NO_EXERCISE
   - RISK_SMOKER
   - RISK_HEAVY_ALCOHOL
   - RISK_POOR_HEALTH

Risk index distribution:
RISK_INDEX
0    220330
1    997097
2    896109
3    314499
4     47230
5      1782
Name: count, dtype: int64


### 8.3 Calculate Prevalence by Risk Index

In [15]:
print("\n[3] Calculating prevalence by risk index...")
prevalence_df = calculate_prevalence_by_risk_level(df)
print("\nPrevalence by Risk Index:")
print(prevalence_df.to_string())

prevalence_df.to_csv(REPORTS_DIR / 'rq4_prevalence_by_risk_index.csv', index=False)
print(f"✓ Saved: {REPORTS_DIR / 'rq4_prevalence_by_risk_index.csv'}")


[3] Calculating prevalence by risk index...

Prevalence by Risk Index:
   Risk_Index  Total_N  Diabetic_N  Prevalence_%  CI_Lower  CI_Upper
0           0   220330        2846      1.291699  1.245391  1.339705
1           1   997097       14595      1.463749  1.440362  1.487510
2           2   896109       12004      1.339569  1.315974  1.363582
3           3   314499        3965      1.260735  1.222332  1.300329
4           4    47230         528      1.117934  1.027007  1.216812
5           5     1782          16      0.897868  0.553419  1.453567
✓ Saved: c:\github\brfss-diabetes-trends\reports\rq4_prevalence_by_risk_index.csv


### 8.4 Calculate Prevalence by Risk Category

In [16]:
print("\n[4] Calculating prevalence by risk category...")
category_df = calculate_prevalence_by_risk_category(df)
print("\nPrevalence by Risk Category:")
print(category_df.to_string())

category_df.to_csv(REPORTS_DIR / 'rq4_prevalence_by_category.csv', index=False)
print(f"✓ Saved: {REPORTS_DIR / 'rq4_prevalence_by_category.csv'}")


[4] Calculating prevalence by risk category...

Prevalence by Risk Category:
  Risk_Category  Total_N  Diabetic_N  Prevalence_%  CI_Lower  CI_Upper
0       0 Risks   220330        2846      1.291699  1.245391  1.339705
1        1 Risk   997097       14595      1.463749  1.440362  1.487510
2       2 Risks   896109       12004      1.339569  1.315974  1.363582
3      3+ Risks   363511        4509      1.240403  1.204934  1.276902
✓ Saved: c:\github\brfss-diabetes-trends\reports\rq4_prevalence_by_category.csv


### 8.5 Test Compounding Effect (Additive vs Multiplicative)

In [17]:
print("\n[5] Testing compounding effect (additive vs multiplicative)...")
compound_df = test_compounding_effect(df)
print("\nCompounding Effect Analysis:")
print(compound_df.to_string())

compound_df.to_csv(REPORTS_DIR / 'rq4_compounding_effect.csv', index=False)
print(f"✓ Saved: {REPORTS_DIR / 'rq4_compounding_effect.csv'}")


[5] Testing compounding effect (additive vs multiplicative)...

Compounding Effect Analysis:
   Risk_Index       N  Observed_Prevalence  Expected_Additive_%  Expected_Multiplicative_%  Additive_Diff  Multiplicative_Diff
0           0  220330             1.291699             1.291699                   1.291699       0.000000         0.000000e+00
1           1  997097             1.463749             1.463749                   1.463749       0.000000         1.734723e-16
2           2  896109             1.339569             1.635800                   1.658331      -0.296231        -3.187621e-01
3           3  314499             1.260735             1.807850                   1.878287      -0.547115        -6.175515e-01
4           4   47230             1.117934             1.979901                   2.126786      -0.861967        -1.008852e+00
5           5    1782             0.897868             2.151951                   2.407354      -1.254083        -1.509487e+00
✓ Saved: c:\githu

### 8.6 Perform Pairwise Statistical Comparisons

In [18]:
print("\n[6] Performing pairwise statistical comparisons...")
comparison_df = perform_pairwise_comparisons(df)
print("\nPairwise Comparisons:")
print(comparison_df.to_string())

comparison_df.to_csv(REPORTS_DIR / 'rq4_pairwise_comparisons.csv', index=False)
print(f"✓ Saved: {REPORTS_DIR / 'rq4_pairwise_comparisons.csv'}")


[6] Performing pairwise statistical comparisons...

Pairwise Comparisons:
            Comparison     Cat1  Cat1_Prevalence_%      Cat2  Cat2_Prevalence_%  Odds_Ratio  Z_Statistic       P_Value  Significant_at_0.05
0    0 Risks vs 1 Risk  0 Risks           1.291699    1 Risk           1.463749    1.135176    -6.150480  7.724877e-10                 True
1   0 Risks vs 2 Risks  0 Risks           1.291699   2 Risks           1.339569    1.037563    -1.757229  7.887871e-02                False
2  0 Risks vs 3+ Risks  0 Risks           1.291699  3+ Risks           1.240403    0.959789     1.703500  8.847446e-02                False
3    1 Risk vs 2 Risks   1 Risk           1.463749   2 Risks           1.339569    0.914011     7.248379  4.217885e-13                 True
4   1 Risk vs 3+ Risks   1 Risk           1.463749  3+ Risks           1.240403    0.845498     9.797500  1.154067e-22                 True
5  2 Risks vs 3+ Risks  2 Risks           1.339569  3+ Risks           1.240403    0.

### 8.7 Analyze Risk Factor Interactions

In [19]:
print("\n[7] Analyzing risk factor interactions...")
interaction_df = analyze_risk_interactions(df, risk_cols)
interaction_df.to_csv(REPORTS_DIR / 'rq4_risk_interactions.csv', index=False)

print("✓ Top 5 synergistic combinations:")
print(interaction_df.head()[['Risk1', 'Risk2', 'Synergy_Index']].to_string())


[7] Analyzing risk factor interactions...
✓ Top 5 synergistic combinations:
                Risk1               Risk2  Synergy_Index
7         RISK_SMOKER  RISK_HEAVY_ALCOHOL       0.338564
9  RISK_HEAVY_ALCOHOL    RISK_POOR_HEALTH       0.192547
3        RISK_OBESITY    RISK_POOR_HEALTH       0.084571
0        RISK_OBESITY    RISK_NO_EXERCISE       0.081062
8         RISK_SMOKER    RISK_POOR_HEALTH       0.042137


### 8.8 Create Visualizations

In [20]:
print("\n[8] Creating visualizations...")
plot_risk_index_analysis(prevalence_df, compound_df, REPORTS_DIR / 'rq4_risk_index_analysis.png')
plot_pairwise_comparisons(comparison_df, category_df, REPORTS_DIR / 'rq4_pairwise_comparisons.png')


[8] Creating visualizations...
✓ Plot saved: c:\github\brfss-diabetes-trends\reports\rq4_risk_index_analysis.png
✓ Plot saved: c:\github\brfss-diabetes-trends\reports\rq4_pairwise_comparisons.png


### 8.9 Conclusions and Hypothesis Testing

In [21]:
print("\n" + "="*70)
print("CONCLUSION")
print("="*70)

# Calculate risk gradient
zero_risk = prevalence_df[prevalence_df['Risk_Index'] == 0]['Prevalence_%'].values[0]
max_risk = prevalence_df[prevalence_df['Risk_Index'] == prevalence_df['Risk_Index'].max()]['Prevalence_%'].values[0]
fold_increase = max_risk / zero_risk if zero_risk > 0 else np.inf

print(f"\n• Zero risk prevalence: {zero_risk:.2f}%")
print(f"• Maximum risk prevalence: {max_risk:.2f}%")
print(f"• Fold increase: {fold_increase:.2f}x")

# Check if multiplicative better than additive
compound_df['Multiplicative_Closer'] = (
    compound_df['Multiplicative_Diff'].abs() < compound_df['Additive_Diff'].abs()
)

mult_better_count = compound_df['Multiplicative_Closer'].sum()
print(f"\n• Multiplicative model fits better: {mult_better_count}/{len(compound_df)} risk levels")

sig_comparisons = comparison_df['Significant_at_0.05'].sum()
print(f"• Significant pairwise differences: {sig_comparisons}/{len(comparison_df)}")

# Synergistic effects
synergistic = interaction_df[interaction_df['Synergistic']]
print(f"• Risk pairs showing synergy: {len(synergistic)}/{len(interaction_df)}")

# Final hypothesis decision
if sig_comparisons == len(comparison_df) and mult_better_count > len(compound_df) / 2:
    print(f"\n✓ REJECT H₀: Strong evidence of synergistic (multiplicative) compounding")
    print(f"  Risk factors interact multiplicatively, not just additively")
elif sig_comparisons > 0:
    print(f"\n✓ PARTIAL REJECT H₀: Evidence of compounding effect")
else:
    print(f"\n✗ FAIL TO REJECT H₀: Insufficient evidence of compounding")

print("\n" + "="*70)


CONCLUSION

• Zero risk prevalence: 1.29%
• Maximum risk prevalence: 0.90%
• Fold increase: 0.70x

• Multiplicative model fits better: 0/6 risk levels
• Significant pairwise differences: 4/6
• Risk pairs showing synergy: 5/10

✓ PARTIAL REJECT H₀: Evidence of compounding effect

